<a href="https://colab.research.google.com/github/PRIYANSHUJAINJECRC/Customer_churn_prediction/blob/main/CustomerChurnPrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Stage A: Problem Understanding & Business Context

Customer churn refers to the phenomenon where customers discontinue their subscription or stop using a company’s services. High churn rates directly impact revenue, customer lifetime value, and long-term business sustainability.

In competitive markets, acquiring new customers is significantly more expensive than retaining existing ones. Therefore, identifying customers who are at high risk of churn enables businesses to take proactive retention measures such as personalized offers, service improvements, or targeted communication.

The objective of this project is to build a machine learning model that predicts customer churn based on demographic, service usage, and contractual features. The model will serve as a decision-support tool for business teams to reduce churn and improve customer retention.

Special attention is given to evaluation metrics and decision thresholds, as false negatives (failing to identify a churning customer) can result in direct revenue loss.


In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


## Stage A: Dataset Loading and Initial Inspection

In this stage, the Telco Customer Churn dataset is loaded using a reliable data source. Initial inspection is performed to verify that the dataset has been imported correctly.

The structure and size of the dataset are examined, and a preview of the first few records is displayed. This step ensures that the data is available in the expected format before proceeding to deeper inspection and preprocessing.

Successful completion of this stage confirms that the dataset is ready for further analysis.




In [2]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "blastchar/telco-customer-churn",
  file_path
)

print("First 5 records:")
print(df.head())
print(df.shape)

100%|██████████| 955k/955k [00:01<00:00, 727kB/s]

First 5 records:
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies    

## Stage B: Dataset Understanding

In this stage, the dataset structure and characteristics are examined to gain a clear understanding of the available data. The data types of each feature are inspected to distinguish between numerical and categorical variables.

Summary statistics are analyzed to observe value ranges, central tendencies, and potential anomalies in numerical features. Special attention is given to identifying columns that may require type conversion or additional cleaning.

The target variable, `Churn`, is analyzed to understand class distribution. The dataset shows class imbalance, with a higher proportion of non-churned customers compared to churned customers. This insight is important for selecting appropriate evaluation metrics and decision thresholds in later stages.

This stage establishes a strong foundation for data cleaning and preprocessing.


In [3]:
print(df.info())
print(df.describe())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [4]:
print(df.isnull().sum())
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True)*100)

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64
Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64


## Stage C: Data Cleaning and Preparation

In this stage, data quality issues are addressed to prepare the dataset for machine learning. The `TotalCharges` feature is converted from a non-numeric format to a numeric type, and invalid entries are handled appropriately.

Missing values introduced during type conversion are imputed using the mean of the column to preserve dataset size and consistency. The target variable, `Churn`, is transformed into a binary numeric format suitable for classification modeling.

By the end of this stage, the dataset is free of missing values and all relevant features are in a machine-readable format, enabling further preprocessing and analysis.


In [5]:
df['TotalCharges']=pd.to_numeric(df['TotalCharges'], errors='coerce')
print(df.isnull().sum())
df['TotalCharges']=df['TotalCharges'].fillna(df['TotalCharges'].mean())
print(df.isnull().sum())
df['Churn']=(df['Churn']=='Yes').astype(int)


customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


## Stage D: Feature Selection and Encoding

In this stage, non-informative identifier columns are removed to prevent noise in the model. The dataset is then separated into input features and the target variable.

All categorical features are transformed into numeric form using encoding techniques so that they can be processed by machine learning algorithms. This step ensures that the feature matrix contains only numerical values while preserving the underlying information.

By the end of this stage, the dataset is fully numeric and structured appropriately for model training.


In [6]:
df = df.drop(columns=['customerID'])
print("DataFrame after dropping 'customerID' column:")


DataFrame after dropping 'customerID' column:


In [7]:
display(df.head())

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [8]:
x=df.drop(columns=['Churn'])
y=df['Churn']

x['gender']=LabelEncoder().fit_transform(x['gender'])
x['Partner']=LabelEncoder().fit_transform(x['Partner'])
x['Dependents']=LabelEncoder().fit_transform(x['Dependents'])
x['PhoneService']=LabelEncoder().fit_transform(x['PhoneService'])
x['MultipleLines']=LabelEncoder().fit_transform(x['MultipleLines'])
x['InternetService']=LabelEncoder().fit_transform(x['InternetService'])
x['OnlineSecurity']=LabelEncoder().fit_transform(x['OnlineSecurity'])
x['OnlineBackup']=LabelEncoder().fit_transform(x['OnlineBackup'])
x['DeviceProtection']=LabelEncoder().fit_transform(x['DeviceProtection'])
x['TechSupport']=LabelEncoder().fit_transform(x['TechSupport'])
x['StreamingTV']=LabelEncoder().fit_transform(x['StreamingTV'])
x['StreamingMovies']=LabelEncoder().fit_transform(x['StreamingMovies'])
x['Contract']=LabelEncoder().fit_transform(x['Contract'])
x['PaperlessBilling']=LabelEncoder().fit_transform(x['PaperlessBilling'])
x['PaymentMethod']=LabelEncoder().fit_transform(x['PaymentMethod'])

print(x.head())
print(y.head())
print(x.shape)
print(y.shape)

   gender  SeniorCitizen  Partner  Dependents  tenure  PhoneService  \
0       0              0        1           0       1             0   
1       1              0        0           0      34             1   
2       1              0        0           0       2             1   
3       1              0        0           0      45             0   
4       0              0        0           0       2             1   

   MultipleLines  InternetService  OnlineSecurity  OnlineBackup  \
0              1                0               0             2   
1              0                0               2             0   
2              0                0               2             2   
3              1                0               2             0   
4              0                1               0             0   

   DeviceProtection  TechSupport  StreamingTV  StreamingMovies  Contract  \
0                 0            0            0                0         0   
1                 

## Stage E: Feature Scaling and Train–Test Split

In this stage, the dataset is divided into training and testing sets to enable unbiased model evaluation. Feature scaling is applied to ensure that all input variables contribute equally during model training.

The scaling parameters are learned exclusively from the training data and then applied to both training and testing sets. This approach prevents data leakage and ensures that model performance metrics reflect real-world behavior.

The target variable remains unchanged, as scaling is not required for classification labels.


In [9]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)
print(y_test.shape)
print(y_train.shape)
print(x_test_scaled)
print(x_train_scaled)

(1409,)
(5634,)
[[-1.02516569 -0.4377492   1.03137591 ...  0.40252821 -1.33162874
  -1.00238817]
 [ 0.97545208 -0.4377492  -0.96957859 ... -1.47094882 -1.31667194
  -0.57263351]
 [-1.02516569 -0.4377492   1.03137591 ...  1.33926673 -1.51277218
  -0.55704266]
 ...
 [ 0.97545208 -0.4377492   1.03137591 ... -1.47094882 -1.49449165
  -0.86751071]
 [-1.02516569 -0.4377492   1.03137591 ... -0.5342103  -0.69513389
   0.29587649]
 [ 0.97545208 -0.4377492  -0.96957859 ...  1.33926673 -1.11392424
  -0.99949115]]
[[-1.02516569e+00 -4.37749204e-01 -9.69578591e-01 ...  1.33926673e+00
  -4.73723375e-04 -4.22098852e-01]
 [-1.02516569e+00 -4.37749204e-01 -9.69578591e-01 ... -1.47094882e+00
   1.07475386e+00  1.25536630e+00]
 [ 9.75452077e-01 -4.37749204e-01  1.03137591e+00 ...  4.02528212e-01
  -1.37649913e+00 -1.00298527e+00]
 ...
 [ 9.75452077e-01 -4.37749204e-01  1.03137591e+00 ...  4.02528212e-01
  -1.45294499e+00 -8.77993070e-01]
 [ 9.75452077e-01  2.28441306e+00 -9.69578591e-01 ...  4.02528212e-

## Stage F: Model Training

In this stage, a Logistic Regression model is trained using the preprocessed and scaled training data. Logistic Regression is chosen as a baseline classification algorithm due to its simplicity, interpretability, and effectiveness for binary classification problems.

The model learns the relationship between customer attributes and churn behavior using the training dataset. After training, predictions are generated on the unseen test data to prepare for performance evaluation in the next stage.

This stage establishes a trained baseline model that will be assessed using appropriate classification metrics.


In [10]:
from sklearn.linear_model import LogisticRegression
model=LogisticRegression()
model.fit(x_train_scaled, y_train)
y_pred=model.predict(x_test_scaled)
print(y_pred[:5,])

[1 0 0 1 0]


## Stage G: Model Evaluation

In this stage, the performance of the trained Logistic Regression model is evaluated using standard classification metrics. Accuracy is calculated to measure overall correctness, while the confusion matrix provides insight into true positives, true negatives, false positives, and false negatives.

A detailed classification report is generated to analyze precision, recall, and F1-score for each class. These metrics are especially important for churn prediction, as failing to identify customers who are likely to churn can result in direct business loss.

This evaluation establishes a baseline understanding of the model’s strengths and weaknesses before further optimization.


In [11]:
accuracy=accuracy_score(y_test, y_pred)
print(accuracy)
confusion=confusion_matrix(y_test, y_pred)
print(confusion)
classification_report=classification_report(y_test, y_pred)
print(classification_report)

0.8161816891412349
[[933 103]
 [156 217]]
              precision    recall  f1-score   support

           0       0.86      0.90      0.88      1036
           1       0.68      0.58      0.63       373

    accuracy                           0.82      1409
   macro avg       0.77      0.74      0.75      1409
weighted avg       0.81      0.82      0.81      1409



## Stage H: Probability Analysis

In this stage, predicted probabilities are generated to understand the confidence of the model’s predictions. Instead of relying only on class labels, probability scores provide insight into how strongly the model believes a customer will churn.

Probabilities close to 0.5 indicate uncertain predictions, while values closer to 0 or 1 represent higher confidence. Analyzing these scores helps identify borderline cases and prepares the groundwork for threshold optimization.

This stage is essential for business-focused decision-making, where adjusting the classification threshold can significantly impact outcomes.


In [17]:
# Predicted probabilities for the positive class (churn = 1)
y_proba = model.predict_proba(x_test_scaled)[:, 1]

# Inspect probabilities
print("First 10 predicted probabilities:")
print(y_proba[:10])

# Compare with predicted labels
print("\nFirst 10 predicted labels:")
print(y_pred[:10])


First 10 predicted probabilities:
[0.64919668 0.06064017 0.00502559 0.57991737 0.0058227  0.30539999
 0.04989031 0.00428333 0.05574349 0.20931115]

First 10 predicted labels:
[1 0 0 1 0 0 0 0 0 0]


## Stage I: Threshold Tuning and Decision Optimization

In this stage, the default classification threshold of 0.5 is adjusted to better align the model’s predictions with business objectives. Instead of directly using predicted class labels, predicted probabilities are converted into labels using a custom threshold.

Lowering the threshold increases the model’s sensitivity to identifying customers who are likely to churn, which can help reduce false negatives. This is particularly important in churn prediction, where failing to identify a potential churner can result in direct revenue loss.

By comparing performance metrics at different thresholds, the trade-off between precision and recall becomes evident. This step highlights how probability-based decision-making can be tailored to real-world business requirements rather than relying solely on default settings.


In [20]:
from sklearn.metrics import confusion_matrix, classification_report

# 1. Choose a custom threshold
custom_threshold = 0.4

# 2. Convert probabilities to class labels using the custom threshold
y_pred_custom = (y_proba >= custom_threshold).astype(int)

# 3. Evaluate performance with the new threshold
print("Custom Threshold:", custom_threshold)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_custom))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_custom))


Custom Threshold: 0.4

Confusion Matrix:
[[867 169]
 [114 259]]

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.84      0.86      1036
           1       0.61      0.69      0.65       373

    accuracy                           0.80      1409
   macro avg       0.74      0.77      0.75      1409
weighted avg       0.81      0.80      0.80      1409



## Stage J: Cross-Validation and Model Robustness

In this stage, cross-validation is performed to assess the stability and generalization capability of the trained Logistic Regression model. Rather than relying on a single train–test split, k-fold cross-validation evaluates the model across multiple subsets of the training data.

Stratified k-fold cross-validation is used to preserve the original class distribution in each fold, which is especially important for churn prediction due to class imbalance. The ROC-AUC metric is selected as the evaluation measure, as it captures the model’s ability to distinguish between churned and non-churned customers across different thresholds.

The mean ROC-AUC score represents overall model performance, while the standard deviation indicates performance consistency across folds. Stable scores with low variance suggest that the model generalizes well and is not overly dependent on a specific data split.

This stage provides confidence that the model’s performance is reliable and suitable for real-world application.


In [21]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
import numpy as np

# Recreate model with fixed settings
cv_model = LogisticRegression(max_iter=1000, random_state=42)

# Stratified K-Fold to preserve class distribution
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Cross-validation using ROC-AUC (recommended for churn)
cv_scores = cross_val_score(
    cv_model,
    x_train_scaled,
    y_train,
    cv=skf,
    scoring="roc_auc"
)

print("Cross-validation ROC-AUC scores:", cv_scores)
print("Mean ROC-AUC:", np.mean(cv_scores))
print("Std ROC-AUC:", np.std(cv_scores))


Cross-validation ROC-AUC scores: [0.8224254  0.84237919 0.83934371 0.84004837 0.85126965]
Mean ROC-AUC: 0.8390932610206822
Std ROC-AUC: 0.00935793175847226


## Stage K: Business Impact Analysis and Final Conclusion

In this stage, the machine learning results are interpreted from a business perspective to understand their practical value. The churn prediction model is designed to act as a decision-support tool that helps organizations proactively identify customers who are at risk of leaving.

By adjusting the classification threshold, the model prioritizes recall over precision, ensuring that a larger proportion of potential churners are identified. From a business standpoint, this approach is often preferred because the cost of retaining an at-risk customer is typically lower than the cost of acquiring a new one.

The insights derived from model interpretation and evaluation can be used by business teams to:
- Design targeted retention campaigns
- Offer personalized discounts or incentives
- Improve customer service for high-risk segments
- Reduce overall churn-related revenue loss

### Key Takeaways
- A complete end-to-end machine learning pipeline was implemented, from data preprocessing to model evaluation.
- Logistic Regression provided a strong baseline model with interpretable results.
- Threshold tuning demonstrated how model decisions can be aligned with business objectives.
- Cross-validation confirmed the model’s stability and robustness.

### Limitations and Future Scope
While the model performs reliably, there are opportunities for improvement:
- Incorporating advanced models such as Random Forest or Gradient Boosting
- Using cost-sensitive learning techniques
- Deploying the model as a web application for real-time predictions
- Integrating explainability tools to enhance stakeholder trust

This project demonstrates the practical application of machine learning to solve a real-world business problem, balancing technical performance with actionable business insights.
